<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/04_progressive_finetune_ms_pd_vicreg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Progressive fine-tuning with ms, pd, and VICReg

The model now knows normal walking. In this notebook we grow its world: we keep training but add the ms and pd clips, and we switch on **VICReg**, an extra regularizer that pushes the three conditions toward separate regions of feature space.

A note on honesty: VICReg is **not** part of the original S-JEPA. The paper prevents collapse with the slow teacher plus centering and sharpening. We add VICReg on top because our goal is classification, and we want the normal, ms, and pd clusters to pull apart. We label it clearly as an extension so nobody mistakes it for the original recipe.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'progressive_timeline.svg')))

## Pick the training videos, without leakage

We must decide now which videos train the model and which are held out, and we must do it by **source id** so clips from one walk never straddle the split. We save that decision to `split_spec.json` so notebooks 05 and 06 reuse the exact same split. This is what makes the later comparison fair.


In [ ]:
import json
from sjepa.data import load_index, grouped_train_test_split

records = load_index(KEYPOINTS_DIR)
train_recs, test_recs = grouped_train_test_split(records, test_size=0.3, seed=42)

split = {
    'train_sources': sorted({r.source_id for r in train_recs}),
    'test_sources': sorted({r.source_id for r in test_recs}),
}
assert not (set(split['train_sources']) & set(split['test_sources']))
(ARTIFACT_DIR / 'split_spec.json').write_text(json.dumps(split, indent=2))
print('train videos:', len(train_recs), '| test videos:', len(test_recs))
print('no source appears in both:', not (set(split['train_sources']) & set(split['test_sources'])))

## Continue training on all three conditions

We load the normal-pretrained weights, then keep training on the full training set with VICReg turned on. Because we have labels during fine-tuning, we use the class-aware VICReg variant, which keeps each condition compact while the variance floor keeps the condition centers from piling up.


In [ ]:
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.train import train_sjepa, load_checkpoint, save_checkpoint
from sjepa.data import SequenceWindowDataset

cfg = get_config()
device = pick_device()
model = build_model(cfg, device=device)
load_checkpoint(ARTIFACT_DIR / 'sjepa_pretrain_normal.pt', model, map_location=device)

ds_all = SequenceWindowDataset(train_recs, cfg.window_frames, cfg.window_stride)
print(f'{len(train_recs)} training videos -> {len(ds_all)} windows across normal/ms/pd')
state = train_sjepa(model, ds_all, cfg, epochs=cfg.finetune_epochs,
                    use_vicreg=True, class_aware_vicreg=True,
                    device=device, log_every=max(1, cfg.finetune_epochs))
save_checkpoint(ARTIFACT_DIR / 'sjepa_finetuned_3class.pt', model, cfg,
                extra={'stage': 'finetune_3class'})

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10,3))
ax[0].plot(state.ce_losses, color='#2b6cb0'); ax[0].set_title('latent cross-entropy')
ax[1].plot(state.vic_losses, color='#38a169'); ax[1].set_title('VICReg term')
for a in ax: a.set_xlabel('step')
plt.tight_layout(); plt.show()

## Does VICReg actually spread the features out?

A quick check: pool the learned features per window and measure their spread. VICReg should keep the per-dimension spread comfortably above zero, meaning the representation did not collapse. We look at it below and visualize the clusters properly in notebook 05.


In [ ]:
import torch, numpy as np
from sjepa.masking import AnatomicalMaskSampler
sampler = AnatomicalMaskSampler(cfg.num_joints, cfg.num_time_tokens)
tm = torch.from_numpy(sampler.target_mask).to(device)
xs = torch.stack([torch.from_numpy(ds_all.windows[i]) for i in range(len(ds_all))]).float().to(device)
with torch.no_grad():
    emb = model.embed(xs, tm).cpu().numpy()
spread = float(emb.std(0).mean())
print('per-dimension std (mean):', round(spread, 4))
assert np.isfinite(emb).all(), 'embeddings went non-finite (training diverged)'
if spread > 1e-3:
    print('Features have healthy spread. Fine-tuned model saved.')
else:
    print('Spread is small. With the full profile (not SJEPA_SMOKE) it opens up;',
          'in a 2-epoch smoke run this is expected.')